<img src="http://dask.readthedocs.io/en/latest/_images/dask_horizontal.svg"
     align="right"
     width="30%"
     alt="Dask logo\">

# Distributed —— 把数据和计算铺到集群上

开头说过，Dask 可以用分布式调度器（distributed scheduler）在多台机器上跑任务。

到目前为止，我们其实一直在用分布式调度器，只是还停留在单机。

当我们执行不带参数的 `Client()` 时，它会尝试寻找一个 Dask 集群：先查看本地 Dask 配置和环境变量里有没有连接信息。如果没有，就会创建一个 `LocalCluster` 实例并使用它。

*在配置里写好连接信息，方便系统管理员把集群交给用户。我们在 [Kubernetes 的 Dask Helm Chart](https://github.com/dask/helm-chart/blob/master/dask/templates/dask-jupyter-deployment.yaml#L46-L48) 里就是这么做的：Chart 会在 Kubernetes 上安装多节点 Dask 集群和 Jupyter 服务，并且预先配置好让 Jupyter 发现这个分布式集群。*


## 本地集群（Local Cluster）

我们自己来看看 `LocalCluster` 对象，弄清它在做什么。


In [1]:
from dask.distributed import LocalCluster, Client

In [2]:
cluster = LocalCluster()
cluster

LocalCluster(7837c710, 'tcp://127.0.0.1:39525', workers=4, threads=4, memory=15.62 GiB)

创建 cluster 对象会启动一个 Dask 调度器以及若干 Dask worker。如果不传参数，它会自动检测系统的 CPU 核数和内存，并据此创建数量合适的 worker。

你也可以自己指定这些参数。先看看文档字符串，了解有哪些选项。

*这些参数也可以传给 `Client`：如果它需要创建 `LocalCluster`，就会原样往下传。*


In [3]:
?LocalCluster

cluster 对象有一些属性和方法，可以用来查看集群信息。例如，用 `get_logs()` 可以拿到调度器和所有 worker 的日志输出。


In [4]:
cluster.get_logs()

{'Cluster': '',
 'Scheduler': "2026-09-17 06:21:11,574 - distributed.scheduler - INFO - State start\n2026-09-17 06:21:11,577 - distributed.scheduler - INFO -   Scheduler at:     tcp://127.0.0.1:39525\n2026-09-17 06:21:11,577 - distributed.scheduler - INFO -   dashboard at:            127.0.0.1:8787\n2026-09-17 06:21:12,527 - distributed.scheduler - INFO - Register worker <WorkerState 'tcp://127.0.0.1:38559', name: 2, status: running, memory: 0, processing: 0>\n2026-09-17 06:21:12,530 - distributed.scheduler - INFO - Starting worker compute stream, tcp://127.0.0.1:38559\n2026-09-17 06:21:12,532 - distributed.scheduler - INFO - Register worker <WorkerState 'tcp://127.0.0.1:44867', name: 0, status: running, memory: 0, processing: 0>\n2026-09-17 06:21:12,532 - distributed.scheduler - INFO - Starting worker compute stream, tcp://127.0.0.1:44867\n2026-09-17 06:21:12,533 - distributed.scheduler - INFO - Register worker <WorkerState 'tcp://127.0.0.1:38625', name: 3, status: running, memory: 0, processing: 0>\n2026-09-17 06:21:12,533 - distributed.scheduler - INFO - Starting worker compute stream, tcp://127.0.0.1:38625\n2026-09-17 06:21:12,536 - distributed.scheduler - INFO - Register worker <WorkerState 'tcp://127.0.0.1:36267', name: 1, status: running, memory: 0, processing: 0>\n2026-09-17 06:21:12,537 - distributed.scheduler - INFO - Starting worker compute stream, tcp://127.0.0.1:36267",
 'tcp://127.0.0.1:36267': '2026-09-17 06:21:12,167 - distributed.worker - INFO -       Start worker at:      tcp://127.0.0.1:36267\n2026-09-17 06:21:12,167 - distributed.worker - INFO -          Listening to:      tcp://127.0.0.1:36267\n2026-09-17 06:21:12,167 - distributed.worker - INFO -           Worker name:                          1\n2026-09-17 06:21:12,167 - distributed.worker - INFO -          dashboard at:            127.0.0.1:34469\n2026-09-17 06:21:12,167 - distributed.worker - INFO - Waiting to connect to:      tcp://127.0.0.1:39525\n2026-09-17 06:21:12,167 - distributed.worker - INFO - -------------------------------------------------\n2026-09-17 06:21:12,167 - distributed.worker - INFO -               Threads:                          1\n2026-09-17 06:21:12,167 - distributed.worker - INFO -                Memory:                   3.90 GiB\n2026-09-17 06:21:12,167 - distributed.worker - INFO -       Local Directory: /tmp/dask-worker-space/worker-jwp5klz1\n2026-09-17 06:21:12,167 - distributed.worker - INFO - -------------------------------------------------\n2026-09-17 06:21:12,537 - distributed.worker - INFO -         Registered to:      tcp://127.0.0.1:39525\n2026-09-17 06:21:12,537 - distributed.worker - INFO - -------------------------------------------------',
 'tcp://127.0.0.1:38559': '2026-09-17 06:21:12,159 - distributed.worker - INFO -       Start worker at:      tcp://127.0.0.1:38559\n2026-09-17 06:21:12,159 - distributed.worker - INFO -          Listening to:      tcp://127.0.0.1:38559\n2026-09-17 06:21:12,159 - distributed.worker - INFO -           Worker name:                          2\n2026-09-17 06:21:12,159 - distributed.worker - INFO -          dashboard at:            127.0.0.1:33621\n2026-09-17 06:21:12,159 - distributed.worker - INFO - Waiting to connect to:      tcp://127.0.0.1:39525\n2026-09-17 06:21:12,159 - distributed.worker - INFO - -------------------------------------------------\n2026-09-17 06:21:12,159 - distributed.worker - INFO -               Threads:                          1\n2026-09-17 06:21:12,159 - distributed.worker - INFO -                Memory:                   3.90 GiB\n2026-09-17 06:21:12,159 - distributed.worker - INFO -       Local Directory: /tmp/dask-worker-space/worker-a3oid73c\n2026-09-17 06:21:12,159 - distributed.worker - INFO - -------------------------------------------------\n2026-09-17 06:21:12,529 - distributed.worker - INFO -         Registered to:      tcp://127.0.0.1:39525\n2026-09-17 06:21:12,530 - distributed.worker - INFO - ----------------------------------------------

我们可以拿到 Dask 仪表盘所在的 URL。


In [5]:
cluster.dashboard_link

'http://127.0.0.1:8787/status'

要让 Dask 使用这个集群，仍然需要创建一个 `Client` 对象。不过既然集群已经建好了，可以直接把它传给 client。


In [6]:
client = Client(cluster)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 4,Total memory: 15.62 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:39525,Workers: 4
Dashboard: http://127.0.0.1:8787/status,Total threads: 4
Started: Just now,Total memory: 15.62 GiB
Comm: tcp://127.0.0.1:44867,Total threads: 1
Dashboard: http://127.0.0.1:46341/status,Memory: 3.90 GiB
Nanny: tcp://127.0.0.1:45919,


In [7]:
del client, cluster

## 通过 SSH 连接远程集群

把工作分发到多台机器的常见方式是 SSH。Dask 有一个集群管理器 `SSHCluster`，会帮你建立 SSH 连接。


```python
from dask.distributed import SSHCluster
```

构造这个集群管理器时，需要传入一组地址（主机名或 IP）。Dask 会 SSH 进去，并尝试在上面启动调度器或 worker。


```python
cluster = SSHCluster(["localhost", "hostA", "hostB"])
cluster
```

创建 `SSHCluster` 时我们给了三个主机名。

列表里的第一台会作为调度器，其余都作为 worker。如果这些机器在同一网络里，把你的本机当成调度器、其他机器当 worker，是很常见的做法。

如果你的服务器在远程（比如在云上），可能也希望调度器放在远程机器上，以免成为网络瓶颈。


## 可伸缩集群

目前见过的两种集群都是固定规模：要么在本地把本机资源用满，要么通过 SSH 明确指定若干其他机器。

有些集群管理器可以增减 worker 数量：既可以在代码里调用 `cluster.scale(n)`（`n` 是想要的 worker 数），也可以让 Dask 动态调整，调用 `cluster.adapt(minimum=1, maximum=100)`，其中 minimum 和 maximum 是你希望 Dask 遵守的上下限。

建议把最小值至少设为 1。Dask 会先在单个 worker 上跑一些任务，估算耗时，再推断还需要多少 worker。按你的环境，申请新 worker 可能要花时间；保持至少 1 个 worker，画像（profiling）就能立刻开始。

目前已有面向 [Kubernetes](https://kubernetes.dask.org/en/latest/)、[Hadoop/Yarn](https://yarn.dask.org/en/latest/)、[云平台](https://cloudprovider.dask.org/en/latest/) 以及 [PBS、SLURM、SGE 等批处理系统](http://jobqueue.dask.org/en/latest/) 的集群管理器。

有权限使用这些资源的用户，可以用这些管理器把 Dask 集群拉起来。如果机构希望提供一个中心服务、让用户按需申请 Dask 集群，还可以使用 [Dask Gateway](https://gateway.dask.org/)。


## 集群组件

一个能工作的 Dask 集群，最低要求是：一个调度器进程，加上至少一个 worker 进程。

这些进程可以用命令行手动启动。先从调度器开始。

```console
$ dask-scheduler                                                               
2022-07-07 14:11:35,661 - distributed.scheduler - INFO - -----------------------------------------------
2022-07-07 14:11:37,405 - distributed.scheduler - INFO - State start
2022-07-07 14:11:37,408 - distributed.scheduler - INFO - -----------------------------------------------
2022-07-07 14:11:37,409 - distributed.scheduler - INFO - Clear task state
2022-07-07 14:11:37,409 - distributed.scheduler - INFO -   Scheduler at:   tcp://10.51.100.80:8786
2022-07-07 14:11:37,409 - distributed.scheduler - INFO -   dashboard at:                     :8787
```

然后可以把 worker 连到调度器正在监听的地址。

```console
$ dask-worker tcp://10.51.100.80:8786 --nworkers=auto
2022-07-07 14:12:53,915 - distributed.nanny - INFO -         Start Nanny at: 'tcp://10.51.100.80:58051'
2022-07-07 14:12:53,922 - distributed.nanny - INFO -         Start Nanny at: 'tcp://10.51.100.80:58052'
2022-07-07 14:12:53,924 - distributed.nanny - INFO -         Start Nanny at: 'tcp://10.51.100.80:58053'
2022-07-07 14:12:53,925 - distributed.nanny - INFO -         Start Nanny at: 'tcp://10.51.100.80:58054'
2022-07-07 14:12:55,222 - distributed.worker - INFO -       Start worker at:   tcp://10.51.100.80:58065
2022-07-07 14:12:55,222 - distributed.worker - INFO -          Listening to:   tcp://10.51.100.80:58065
2022-07-07 14:12:55,223 - distributed.worker - INFO -          dashboard at:         10.51.100.80:58068
2022-07-07 14:12:55,223 - distributed.worker - INFO - Waiting to connect to:    tcp://10.51.100.80:8786
2022-07-07 14:12:55,223 - distributed.worker - INFO - -------------------------------------------------
2022-07-07 14:12:55,223 - distributed.worker - INFO -               Threads:                          3
2022-07-07 14:12:55,223 - distributed.worker - INFO -                Memory:                   4.00 GiB
2022-07-07 14:12:55,224 - distributed.worker - INFO -       Local Directory: /Users/jtomlinson/Projects/dask/dask-tutorial/dask-worker-space/worker-hlvac6m5
2022-07-07 14:12:55,225 - distributed.worker - INFO - -------------------------------------------------
2022-07-07 14:12:55,227 - distributed.worker - INFO -       Start worker at:   tcp://10.51.100.80:58066
2022-07-07 14:12:55,227 - distributed.worker - INFO -          Listening to:   tcp://10.51.100.80:58066
2022-07-07 14:12:55,227 - distributed.worker - INFO -          dashboard at:         10.51.100.80:58070
2022-07-07 14:12:55,227 - distributed.worker - INFO - Waiting to connect to:    tcp://10.51.100.80:8786
2022-07-07 14:12:55,227 - distributed.worker - INFO - -------------------------------------------------
2022-07-07 14:12:55,227 - distributed.worker - INFO -               Threads:                          3
2022-07-07 14:12:55,228 - distributed.worker - INFO -                Memory:                   4.00 GiB
2022-07-07 14:12:55,228 - distributed.worker - INFO -       Local Directory: /Users/jtomlinson/Projects/dask/dask-tutorial/dask-worker-space/worker-e1suf_7o
2022-07-07 14:12:55,229 - distributed.worker - INFO - -------------------------------------------------
2022-07-07 14:12:55,231 - distributed.worker - INFO -       Start worker at:   tcp://10.51.100.80:58063
2022-07-07 14:12:55,233 - distributed.worker - INFO -          Listening to:   tcp://10.51.100.80:58063
2022-07-07 14:12:55,233 - distributed.worker - INFO -          dashboard at:         10.51.100.80:58067
2022-07-07 14:12:55,233 - distributed.worker - INFO - Waiting to connect to:    tcp://10.51.100.80:8786
2022-07-07 14:12:55,233 - distributed.worker - INFO - -------------------------------------------------
2022-07-07 14:12:55,234 - distributed.worker - INFO -               Threads:                          3
2022-07-07 14:12:55,234 - distributed.worker - INFO -                Memory:                   4.00 GiB
2022-07-07 14:12:55,235 - distributed.worker - INFO -       Local Directory: /Users/jtomlinson/Projects/dask/dask-tutorial/dask-worker-space/worker-oq39ihb4
2022-07-07 14:12:55,236 - distributed.worker - INFO - -------------------------------------------------
2022-07-07 14:12:55,246 - distributed.worker - INFO -         Registered to:    tcp://10.51.100.80:8786
2022-07-07 14:12:55,246 - distributed.worker - INFO - -------------------------------------------------
2022-07-07 14:12:55,249 - distributed.core - INFO - Starting established connection
2022-07-07 14:12:55,264 - distributed.worker - INFO -         Registered to:    tcp://10.51.100.80:8786
2022-07-07 14:12:55,264 - distributed.worker - INFO - -------------------------------------------------
2022-07-07 14:12:55,267 - distributed.worker - INFO -         Registered to:    tcp://10.51.100.80:8786
2022-07-07 14:12:55,267 - distributed.core - INFO - Starting established connection
2022-07-07 14:12:55,267 - distributed.worker - INFO - -------------------------------------------------
2022-07-07 14:12:55,269 - distributed.core - INFO - Starting established connection
2022-07-07 14:12:55,273 - distributed.worker - INFO -       Start worker at:   tcp://10.51.100.80:58064
2022-07-07 14:12:55,273 - distributed.worker - INFO -          Listening to:   tcp://10.51.100.80:58064
2022-07-07 14:12:55,273 - distributed.worker - INFO -          dashboard at:         10.51.100.80:58069
2022-07-07 14:12:55,273 - distributed.worker - INFO - Waiting to connect to:    tcp://10.51.100.80:8786
2022-07-07 14:12:55,274 - distributed.worker - INFO - -------------------------------------------------
2022-07-07 14:12:55,274 - distributed.worker - INFO -               Threads:                          3
2022-07-07 14:12:55,275 - distributed.worker - INFO -                Memory:                   4.00 GiB
2022-07-07 14:12:55,275 - distributed.worker - INFO -       Local Directory: /Users/jtomlinson/Projects/dask/dask-tutorial/dask-worker-space/worker-zfie55ku
2022-07-07 14:12:55,276 - distributed.worker - INFO - -------------------------------------------------
2022-07-07 14:12:55,299 - distributed.worker - INFO -         Registered to:    tcp://10.51.100.80:8786
2022-07-07 14:12:55,300 - distributed.worker - INFO - -------------------------------------------------
2022-07-07 14:12:55,302 - distributed.core - INFO - Starting established connection
```

然后在 Python 里可以把 client 连到这个集群并提交任务。

```python
>>> from dask.distributed import Client
>>> client = Client("tcp://10.51.100.80:8786")
>>> client.submit(lambda: 1+1)
```

也可以在 Python 里直接导入集群组件并创建它们。


In [8]:
from dask.distributed import Scheduler, Worker, Client

async with Scheduler() as scheduler:
    async with Worker(scheduler.address) as worker:
        async with Client(scheduler.address, asynchronous=True) as client:
            print(await client.submit(lambda: 1 + 1))

2


/usr/share/miniconda/envs/dask-tutorial/lib/python3.10/site-packages/distributed/node.py:182: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 37055 instead
  warnings.warn(


大多数时候我们不必亲手创建这些组件，而是交给集群管理器对象去做。但在某些场景下，自己手动搭一个集群会很有用。

你可能还会时不时看到 `Nanny` 进程。它是 worker 的包装，负责在进程被杀掉后重启。通过 CLI 运行 `dask-worker` 时，会自动为我们创建一个 nanny。


## 集群网络

默认情况下，Dask 用一套基于 TCP 的自定义远程过程调用协议在进程间通信。调度器和 worker 都会监听 TCP 端口。

启动调度器时，它通常监听 `8786` 端口。创建 worker 时，它会监听一个随机的高位端口，并在首次连接时把这个端口告诉调度器。

调度器维护着所有 worker 及其地址的列表，worker 也可以访问这份信息，因此调度器和任意 worker 随时都能连到其他 worker。连接在不用时会自动关闭。

`Client` 永远只连接调度器，与 worker 的通信都经过调度器。这意味着部署 Dask 集群时，调度器和 worker 通常必须在同一网络中，并能通过 IP 和端口直接互访；而 client 只要能访问调度器的通信端口，放在哪里都可以。常见做法是用防火墙规则或负载均衡，只开放调度器端口。

Dask 也支持其他网络协议，例如 [TLS](https://distributed.dask.org/en/stable/tls.html)、[websockets](https://distributed.dask.org/en/stable/protocol.html) 和 [UCX](https://docs.rapids.ai/api/dask-cuda/nightly/examples/ucx.html)。


### 用 TLS/SSL 做安全通信

如果运行在不可信环境中，Dask 集群组件可以用证书做双向认证和加密通信。你可以为调度器、worker 和 client 生成并分发证书，也可以生成临时凭证。

像 `dask-cloudprovider` 这类集群管理器，在公有云上把集群暴露到互联网时，往往会自动启用 TLS 并生成一次性证书。


In [9]:
from dask.distributed import Scheduler, Worker, Client
from distributed.security import Security

security = Security.temporary()

async with Scheduler(security=security) as scheduler:
    async with Worker(scheduler.address, security=security) as worker:
        async with Client(
            scheduler.address, security=security, asynchronous=True
        ) as client:
            print(await client.submit(lambda: 1 + 1))

2


/usr/share/miniconda/envs/dask-tutorial/lib/python3.10/site-packages/distributed/node.py:182: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 36895 instead
  warnings.warn(


### Websockets

Dask 也可以改用 websocket 而不是 TCP。这样做会有非常小的性能开销，但仪表盘和通信会走同一个端口，从而可以被 nginx 这类七层代理反向代理。在不能直接暴露端口、但可以代理 Web 服务的部署场景里，这是必要的。


### UCX

在具备 InfiniBand 或 NVLink 等高性能网络的系统上，Dask 还可以使用 [UCX](https://openucx.org/)。它提供统一的通信协议，并自动升级到当前最快的硬件。这对 InfiniBand 上的 HPC 系统、或多 GPU worker 的机器上的性能至关重要。
